# Inter-Annotator Agreement

Computes raw agreement, Cohen's κ, and Krippendorff's α between annotators on the manual classification labels.

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
import krippendorff
import matplotlib.pyplot as plt

In [ ]:
ANNOTATOR1_PATH = "../../../data/annotator_agreement/manual_labels_annotator1.csv"
ANNOTATOR2_PATH   = "../../../data/annotator_agreement/manual_labels_annotator2.csv"

annotator1 = pd.read_csv(ANNOTATOR1_PATH)[["original_index", "manual_label"]].rename(columns={"manual_label": "label_annotator1"})
annotator2   = pd.read_csv(ANNOTATOR2_PATH)[["original_index", "manual_label"]].rename(columns={"manual_label": "label_annotator2"})

df = annotator1.merge(annotator2, on="original_index")

# Exclude items where either annotator skipped
df = df[(df["label_annotator1"] != "skip") & (df["label_annotator2"] != "skip")].copy()

print(f"Items after merging (excl. skips): {len(df)}")
print()
print("Annotator 1 label distribution:")
print(df["label_annotator1"].value_counts())
print()
print("annotator2 label distribution:")
print(df["label_annotator2"].value_counts())

In [ ]:
df.loc[:,'label_annotator1'] = df.label_annotator1.apply(lambda x: 'valid' if x in ['cleaned','unchanged'] else x)
df.loc[:,'label_annotator2'] = df.label_annotator2.apply(lambda x: 'valid' if x in ['cleaned','unchanged'] else x)
df

In [ ]:
import sys
sys.path.insert(0, '../..')
from libs.metrics.agreement import compute_agreement, print_agreement_summary

stats = compute_agreement(df, 'label_annotator1', 'label_annotator2')
print_agreement_summary(stats)


In [ ]:
from libs.metrics.agreement import plot_confusion

categories = ['valid', 'fixed_dict', 'empty', 'invalid', 'refused']
fig, ax = plot_confusion(
    df, 'label_annotator1', 'label_annotator2',
    labels=categories,
    title='Confusion Matrix\n(rows = Annotator 1, cols = Annotator 2)',
)
ax.set_xlabel('Annotator 2')
ax.set_ylabel('Annotator 1')
fig.show()


# Accuracy

In [ ]:
import pandas as pd
import json

In [ ]:
SAMPLE_PATH   = "../../../data/annotator_agreement/sample_100.csv"

In [ ]:
df_sample = pd.read_csv(SAMPLE_PATH)
df_sample.head(2)

In [ ]:
cols = ['original_index','valid_flag','manual_label']
annotator1 = pd.read_csv(ANNOTATOR1_PATH)[cols]
annotator2  = pd.read_csv(ANNOTATOR2_PATH)[cols]

In [ ]:
annotator1.loc[:, 'manual_label_clean'] = annotator1.manual_label.apply(lambda x: 'valid' if x in ['unchanged','cleaned'] else x)
annotator2.loc[:, 'manual_label_clean'] = annotator2.manual_label.apply(lambda x: 'valid' if x in ['unchanged','cleaned'] else x)

In [ ]:
df_annotations = annotator1.drop(columns=['manual_label']).merge(annotator2.drop(columns=['manual_label','valid_flag']), on="original_index", suffixes=("_annotator1", "_annotator2"))
df_annotations.loc[:, 'agreement'] = df_annotations.apply(lambda row: row['manual_label_clean_annotator1'] == row['manual_label_clean_annotator2'], axis=1)
df_annotations

In [ ]:
df_annotations.query("agreement == False")

In [ ]:
new_labels  = { 824376:'valid',
               822530:'refused',
               620633:'valid',
               219509:'invalid',
               698887:'invalid',
               704971:'valid',
               560751:'invalid', # maybe fixed
               741207:'invalid',
               763064:'fixed_dict',
               347891:'fixed_dict',
               332597:'fixed_dict',
               }

In [ ]:
df_annotations.loc[:,'annotation'] = df_annotations.apply(lambda row: row['manual_label_clean_annotator1'] if row['agreement']==True else new_labels[row['original_index']], axis=1)
df_annotations

In [ ]:
df_annotations_eval = df_annotations[['valid_flag','annotation']].copy()
df_annotations_eval.rename(columns={'annotation':'true', 'valid_flag':'pred'}, inplace=True)
df_annotations_eval.loc[:, 'pred'] = df_annotations_eval.pred.apply(lambda x: 'valid' if x in ['unchanged','cleaned'] else x)
df_annotations_eval

In [ ]:
from libs.metrics.agreement import compute_classification_metrics
from sklearn.metrics import (
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
)

stats = compute_classification_metrics(
    df_annotations_eval.rename(columns={'true': 'manual_label'}),
    pred_col='pred',
)
y_true = df_annotations_eval['true']
y_pred = df_annotations_eval['pred']
mask = y_true.notna() & y_pred.notna()
y_true, y_pred = y_true[mask], y_pred[mask]
labels = stats['labels']

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average='macro', zero_division=0,
)
p_w, r_w, f_w, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average='weighted', zero_division=0,
)

metrics = pd.DataFrame({
    'metric': [
        'accuracy', 'balanced_accuracy',
        'precision_macro', 'recall_macro', 'f1_macro',
        'precision_weighted', 'recall_weighted', 'f1_weighted',
        'cohen_kappa', 'matthews_corrcoef',
    ],
    'value': [
        stats['accuracy'], balanced_accuracy_score(y_true, y_pred),
        p_macro, r_macro, f_macro,
        p_w, r_w, f_w,
        cohen_kappa_score(y_true, y_pred),
        matthews_corrcoef(y_true, y_pred),
    ],
})
metrics


In [ ]:
from sklearn.metrics import classification_report

# Per class:
report = classification_report(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).T
report_df

In [ ]:
cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=labels),
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels],
)

cm